# YOLOv26 — Benchmark on OOD Dataset (22 classes)

Train all YOLOv26 variants (n/s/m/l) on the OOD sidewalk obstacle dataset, then compare:
- **mAP@0.5** and **mAP@0.5:0.95**
- **Precision**, **Recall**, **F1 Score**
- **Inference speed** (ms/img)
- **Model size** (MB) and **parameters** (M)
- **Per-class AP**

In [1]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [2]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 100
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov26.csv"
PERCLASS_CSV = "benchmark_yolov26_perclass.csv"

CLASS_NAMES = [
    'bench', 'bicycle', 'bus', 'bus_stop', 'car', 'crutch', 'curb', 'dog',
    'fire_hydrant', 'motorcycle', 'person', 'pole', 'spherical_roadblock',
    'stairs', 'stop_sign', 'street_light', 'traffic_light', 'train', 'tree',
    'truck', 'warning_column', 'waste_container'
]

MODELS = [
    "yolo26n.pt",
    "yolo26s.pt",
    "yolo26m.pt",
   
]

In [3]:
def benchmark_model(model_name):
    """Train a model on OOD and return benchmark metrics."""
    print(f"\n{'='*60}")
    print(f"  BENCHMARK: {model_name}")
    print(f"{'='*60}")

    model = YOLO(model_name)

    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        name=f"{model_name.replace('.pt', '')}_ood",
        patience=PATIENCE,
        save=True,
        plots=True,
    )

    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))

    metrics = best_model.val(data=DATA_YAML, split="test", device=DEVICE, imgsz=IMG_SIZE)

    test_img_dir = Path("../dataset/test/images")
    _ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    test_images = [p for p in test_img_dir.glob("*.*") if p.suffix.lower() in _ext][:200]
    if not test_images:
        raise FileNotFoundError(f"No test images under {test_img_dir.resolve()}")
    latencies = []
    for _ in range(3):
        best_model(str(test_images[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    for img_path in test_images:
        t0 = time.perf_counter()
        best_model(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)

    size_mb = best_path.stat().st_size / 1e6 if best_path.exists() else 0.0
    params_m = sum(p.numel() for p in best_model.model.parameters()) / 1e6
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    row = {
        "model":         model_name.replace(".pt", ""),
        "mAP@0.5":       round(float(metrics.box.map50), 4),
        "mAP@0.5:0.95":  round(float(metrics.box.map), 4),
        "precision":     round(p, 4),
        "recall":        round(r, 4),
        "F1":            round(f1, 4),
        "speed_ms/img":  round(float(np.mean(latencies)), 2),
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }

    per_class = {}
    ap50_per_class = metrics.box.ap50
    for i, ap in enumerate(ap50_per_class):
        per_class[CLASS_NAMES[i]] = round(float(ap), 4)

    del model, best_model
    gc.collect()
    torch.cuda.empty_cache()

    return row, per_class

In [4]:
rows = []
all_per_class = {}

for model_name in MODELS:
    try:
        row, per_class = benchmark_model(model_name)
        rows.append(row)
        all_per_class[row["model"]] = per_class

        print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
        print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
        print(f"  Precision     : {row['precision']}")
        print(f"  Recall        : {row['recall']}")
        print(f"  F1            : {row['F1']}")
        print(f"  Speed         : {row['speed_ms/img']} ms/img")
        print(f"  Size          : {row['size_MB']} MB")
        print(f"  Params        : {row['params_M']} M")
    except Exception as e:
        print(f"  SKIPPED {model_name}: {e}")
        gc.collect()
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(RESULTS_CSV, index=False)

df_pc = pd.DataFrame(all_per_class).T
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV}")
print(f"Saved -> {PERCLASS_CSV}")


  BENCHMARK: yolo26n.pt
New https://pypi.org/project/ultralytics/8.4.35 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_sc

RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(RESULTS_CSV)

print("=" * 60)
print("  YOLOv26 BENCHMARK — OOD Dataset (22 classes)")
print("=" * 60)

styled = (
    df.style
    .set_caption("YOLOv26 Benchmark — OOD Dataset")
    .format({
        "mAP@0.5":      "{:.4f}",
        "mAP@0.5:0.95": "{:.4f}",
        "precision":    "{:.4f}",
        "recall":       "{:.4f}",
        "F1":           "{:.4f}",
        "speed_ms/img": "{:.1f} ms",
        "size_MB":      "{:.1f} MB",
        "params_M":     "{:.1f} M",
    })
    .highlight_max(subset=["mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1"], color="#2d6a2e")
    .highlight_min(subset=["speed_ms/img", "size_MB", "params_M"], color="#1a5276")
    .set_properties(**{"text-align": "center", "font-size": "13px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("padding", "8px")]},
    ])
    .hide(axis="index")
)
display(styled)

In [ ]:
df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)

print("Per-class mAP@0.5 across YOLOv26 variants")
print("-" * 60)

styled_pc = (
    df_pc.style
    .set_caption("Per-Class mAP@0.5 — YOLOv26 Benchmark")
    .format("{:.4f}")
    .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
    .set_properties(**{"text-align": "center", "font-size": "12px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "11px"), ("padding", "6px")]},
    ])
)
display(styled_pc)

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_CSV)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(df["model"], df["mAP@0.5"], color="#2d6a2e")
axes[0].set_xlabel("mAP@0.5")
axes[0].set_title("Accuracy (mAP@0.5)")
axes[0].set_xlim(0, 1)

axes[1].barh(df["model"], df["speed_ms/img"], color="#1a5276")
axes[1].set_xlabel("ms / image")
axes[1].set_title("Inference Speed")

axes[2].barh(df["model"], df["size_MB"], color="#c0392b")
axes[2].set_xlabel("MB")
axes[2].set_title("Model Size")

plt.suptitle("YOLOv26 Benchmark — OOD Dataset", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("benchmark_yolov26_chart.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os, random

best_model_name = df.loc[df["mAP@0.5"].idxmax(), "model"]
best_path = f"../runs/detect/{best_model_name}_ood/weights/best.pt"
print(f"Best model: {best_model_name} — loading {best_path}")

model = YOLO(best_path)

test_img_dir = "../dataset/test/images"
images = random.sample(os.listdir(test_img_dir), min(9, len(os.listdir(test_img_dir))))

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
for ax, img_name in zip(axes.flatten(), images):
    img_path = os.path.join(test_img_dir, img_name)
    results = model(img_path, conf=0.25, verbose=False)[0]
    img_plot = results.plot()[:, :, ::-1]
    ax.imshow(img_plot)
    ax.axis("off")
    ax.set_title(f"{len(results.boxes)} detections", fontsize=10)

plt.suptitle(f"{best_model_name} (Best) — Detections on OOD Test Set", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig("benchmark_yolov26_detections.png", dpi=150, bbox_inches="tight")
plt.show()